# Stage 4 — Phase 2: Data Preparation

Builds on Phase 0-1 (`Stage4_Phase0-1_Data_Inspection.ipynb`) — assumes datasets are present,
verified, and integrity-checked. Starts with 2.1: locking the Gupta test split.

## 0. Configuration + mount

In [ ]:
DRIVE_ROOT   = "/content/drive/MyDrive/pid_project/data"
KAGGLE_DIR   = "kaggle_pid_symbols"
GUPTA_DIR    = "gupta_pid"
PID2GRAPH_DIR= "pid2graph"
DATA_VERSION = "data-v1"

from google.colab import drive
drive.mount('/content/drive')

import os, json, hashlib
from pathlib import Path
from PIL import Image

ROOT = Path(DRIVE_ROOT)
gupta_p = ROOT / GUPTA_DIR
print("Gupta root:", gupta_p, "| exists:", gupta_p.exists())

## 2.2a Visually verify candidate class names (before trusting classes.json)

Found an external `dataset.yaml` from the dataset author's own GitHub repo
(`ch-hristov/p-id-symbols`) with candidate class names — but it maps index 0 to
`"Not_used"`, while index 0 is our **most frequent** class (3,832 instances). That's a
contradiction, so don't trust it blindly. Crop a few real instances per class ID and
check by eye whether the claimed name plausibly matches the symbol shape.

In [ ]:
CANDIDATE_NAMES = {
    0: "Not_used", 1: "Gate_Valve", 2: "Ball_Valve", 3: "Globe_valve_NO",
    4: "Gate_valve_NO", 5: "Globe_valve_NO", 6: "Butterfly_valve", 7: "Plug valve",
    8: "Check_valve", 9: "Diaphragm_valve", 10: "Needle_valve",
    11: "Half_Filled_Gate_Valve", 12: "Gate_Valve_NC", 13: "Globle_valve_NC",
    14: "Control_Valve", 15: "Rotary_Valve", 16: "Ball_valve_NC", 17: "Paddle_blind",
    18: "Spectacle_blind_Closed", 19: "Spectacle_blind_Open", 20: "Reducer",
    21: "Flange_or_Nozzle", 22: "Rupture_disk", 23: "Pipe_Insulation_or_Tracing",
    24: "Flow_Arrow", 25: "sight_glass", 26: "Instrument_Field", 27: "Instrument_Field",
    28: "Instrument_Panel", 29: "Instrument_Aux_Panel", 30: "box",
    31: "Instrument_Panel",
}

kaggle_p = ROOT / KAGGLE_DIR

def crops_for_class(cls_id, n=3):
    """Find n example crops for a given class ID by scanning label files."""
    found = []
    for lbl in (kaggle_p / "labels").glob("*.txt"):
        for line in lbl.read_text().splitlines():
            parts = line.strip().split()
            if not parts or int(parts[0]) != cls_id:
                continue
            cx, cy, w, h = (float(v) for v in parts[1:5])
            img_path = kaggle_p / "images" / f"{lbl.stem}.jpg"
            found.append((img_path, cx, cy, w, h))
            if len(found) >= n:
                return found
    return found

def show_class_crops(cls_id, n=3, pad=1.5):
    name = CANDIDATE_NAMES.get(cls_id, "?")
    print(f"\n=== class {cls_id} — candidate name: {name} ===")
    for img_path, cx, cy, w, h in crops_for_class(cls_id, n):
        img = Image.open(img_path)
        W, H = img.size
        cx, cy, w, h = cx * W, cy * H, w * W * pad, h * H * pad
        box = (max(0, cx - w/2), max(0, cy - h/2), min(W, cx + w/2), min(H, cy + h/2))
        display(img.crop(box))

# Check a handful spanning different claimed categories
for cls_id in [0, 1, 6, 21]:
    show_class_crops(cls_id)

## 2.1 Lock the Gupta test split

Per CLAUDE.md hard rule #1: the 20-sheet test split is **never opened during training or
tuning** — touched exactly once, at final scoring. This cell freezes `test_ids.json` to
Drive as the single source of truth. If the file already exists, it verifies against it
rather than silently overwriting — the whole point is that this file doesn't drift.

In [2]:
raw = gupta_p / "PID_Dataset" / "0__raw_data"
train_ids = sorted(p.stem for p in (raw / "sheets" / "train").iterdir())
test_ids  = sorted(p.stem for p in (raw / "sheets" / "test").iterdir())

assert len(train_ids) == 72, f"expected 72 train sheets, got {len(train_ids)}"
assert len(test_ids) == 20, f"expected 20 test sheets, got {len(test_ids)}"

overlap = set(train_ids) & set(test_ids)
assert not overlap, f"train/test overlap found: {overlap}"

split_record = {
    "data_version": DATA_VERSION,
    "train_ids": train_ids,
    "test_ids": test_ids,
}

test_ids_path = ROOT / "test_ids.json"

if test_ids_path.exists():
    existing = json.loads(test_ids_path.read_text())
    assert existing["test_ids"] == test_ids, (
        "test_ids.json already exists and DIFFERS from the current split — "
        "this file is frozen per CLAUDE.md hard rule #1. Investigate before overwriting; "
        "do not just re-run and accept a new split."
    )
    print(f"test_ids.json already exists and matches current split — unchanged. ({test_ids_path})")
else:
    test_ids_path.write_text(json.dumps(split_record, indent=2))
    print(f"Wrote frozen split to {test_ids_path}")

print(f"\ntrain: {len(train_ids)} | test: {len(test_ids)} | overlap: {len(overlap)}")
print("✓ 72/20 split locked, zero overlap")

Wrote frozen split to /content/drive/MyDrive/pid_project/data/test_ids.json

train: 72 | test: 20 | overlap: 0
✓ 72/20 split locked, zero overlap
